# 02 – Khám phá dữ liệu (EDA)

**Phần 1:** Phân phối biến mục tiêu `Appliances` và giá trị ngoại lệ.

**Phần 2:** Xu hướng tiêu thụ theo thời gian (giờ, ngày trong tuần, theo tuần).

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.append(str(ROOT))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from src.config import TARGET
from src.data_loader import load_raw_data
from src.visualization import save_fig, set_style

set_style()
df = load_raw_data()
y = df[TARGET]
df.shape

## Phần 1. Phân phối biến mục tiêu
### 1.1 Histogram và boxplot

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
sns.histplot(y, bins=80, ax=axes[0])
axes[0].axvline(y.mean(), color="#e4572e", linestyle="--", label=f"Trung bình = {y.mean():.1f}")
axes[0].axvline(y.median(), color="#29a19c", linestyle="--", label=f"Trung vị = {y.median():.0f}")
axes[0].set_title("Phân phối Appliances (Wh)")
axes[0].set_xlabel("Appliances (Wh)")
axes[0].legend()

sns.boxplot(x=y, ax=axes[1])
axes[1].set_title("Boxplot Appliances (Wh)")
axes[1].set_xlabel("Appliances (Wh)")
fig.tight_layout()
save_fig(fig, "02_target_distribution")
plt.show()

In [ ]:
print("Skewness:", round(y.skew(), 2))
y.quantile([0.5, 0.75, 0.9, 0.95, 0.99]).to_frame("Wh")

**Nhận xét:**
- Phân phối **lệch phải rất mạnh** (skewness ≈ 3.4): phần lớn thời gian thiết bị tiêu thụ 40–100 Wh,
  nhưng có đuôi dài tới 1080 Wh.
- Trung bình (~98 Wh) lớn hơn hẳn trung vị (60 Wh) do bị các đỉnh cao kéo lên.
- 90% mẫu ≤ 196 Wh, nhưng 1% mẫu cao nhất vượt 576 Wh.

### 1.2 Giá trị ngoại lệ theo quy tắc IQR

In [ ]:
q1, q3 = y.quantile([0.25, 0.75])
iqr = q3 - q1
upper = q3 + 1.5 * iqr
n_out = (y > upper).sum()
print(f"Q1 = {q1:.0f}, Q3 = {q3:.0f}, IQR = {iqr:.0f}")
print(f"Ngưỡng trên = {upper:.0f} Wh")
print(f"Số mẫu vượt ngưỡng: {n_out} ({n_out / len(y):.2%})")

**Quyết định xử lý ngoại lệ:**
- Có ~10,8% mẫu vượt ngưỡng IQR (175 Wh). Tỷ lệ này **quá lớn** để coi là lỗi đo.
- Về bản chất vật lý, các giá trị này là **những lúc nhiều thiết bị cùng hoạt động** (nấu ăn, giặt, ...),
  chính là hiện tượng mô hình cần dự đoán.
- → **Không xóa** ngoại lệ ở biến mục tiêu. Thay vào đó xem xét **biến đổi log** để giảm ảnh hưởng của đuôi dài
  khi huấn luyện, và báo cáo thêm **MAE** (ít nhạy với ngoại lệ hơn RMSE).

### 1.3 Biến đổi log1p

In [ ]:
y_log = np.log1p(y)
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
sns.histplot(y, bins=80, ax=axes[0])
axes[0].set_title(f"Gốc (skew = {y.skew():.2f})")
axes[0].set_xlabel("Appliances (Wh)")
sns.histplot(y_log, bins=80, ax=axes[1], color="#29a19c")
axes[1].set_title(f"Sau log1p (skew = {y_log.skew():.2f})")
axes[1].set_xlabel("log(1 + Appliances)")
fig.tight_layout()
save_fig(fig, "02_target_log_transform")
plt.show()

**Nhận xét:** Sau `log1p`, độ lệch giảm từ ~3.39 xuống ~1.19 – phân phối cân đối hơn nhiều.
Nhóm sẽ thực nghiệm huấn luyện mô hình trên `log1p(Appliances)` rồi đổi ngược (`expm1`) khi đánh giá,
để metric vẫn tính trên đơn vị Wh gốc và so sánh công bằng với mô hình không dùng log.

### 1.4 Biến `lights`

In [ ]:
print("Tỷ lệ lights = 0:", f"{(df['lights'] == 0).mean():.1%}")
print("Tương quan lights – Appliances:", round(df["lights"].corr(y), 3))
df["lights"].value_counts().sort_index().to_frame("số mẫu")

`lights` bằng 0 trong ~77% thời gian và chỉ nhận vài giá trị rời rạc (0, 10, 20, ...). Tương quan với
`Appliances` dương nhưng yếu (~0.2). Đây là tín hiệu "có người đang ở nhà/hoạt động", nên vẫn giữ làm đặc trưng.

## Phần 2. Xu hướng theo thời gian
Dùng hàm `add_time_features` trong `src/features.py` để trích xuất giờ, thứ trong tuần.

In [ ]:
from src.features import add_time_features

dft = add_time_features(df)
DAY_NAMES = ["T2", "T3", "T4", "T5", "T6", "T7", "CN"]

### 2.1 Tiêu thụ trung bình theo giờ trong ngày

In [ ]:
by_hour = dft.groupby("hour")[TARGET].agg(["mean", "median"])
fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(by_hour.index, by_hour["mean"], marker="o", label="Trung bình")
ax.plot(by_hour.index, by_hour["median"], marker="s", label="Trung vị")
ax.set_xticks(range(24))
ax.set_title("Điện năng thiết bị theo giờ trong ngày")
ax.set_xlabel("Giờ")
ax.set_ylabel("Appliances (Wh)")
ax.legend()
fig.tight_layout()
save_fig(fig, "02_by_hour")
plt.show()

**Nhận xét:**
- Ban đêm (0h–6h) tiêu thụ thấp và ổn định (~50 Wh) – chỉ còn các thiết bị nền như tủ lạnh.
- Tăng mạnh từ 7h, đạt đỉnh phụ quanh 11h (~133 Wh) và **đỉnh chính lúc 18h (~190 Wh)** – giờ nấu ăn tối.
- Giờ trong ngày là đặc trưng rất mạnh → cần đưa vào mô hình (qua `hour`/`nsm` và mã hóa chu kỳ sin/cos).

### 2.2 Theo thứ trong tuần

In [ ]:
by_day = dft.groupby("day_of_week")[TARGET].mean()
fig, ax = plt.subplots(figsize=(8, 3.8))
ax.bar(DAY_NAMES, by_day.values)
ax.set_title("Điện năng trung bình theo thứ trong tuần")
ax.set_ylabel("Appliances (Wh)")
fig.tight_layout()
save_fig(fig, "02_by_weekday")
plt.show()
print(dft.groupby("is_weekend")[TARGET].mean().rename({0: "Ngày thường", 1: "Cuối tuần"}).round(1))

**Nhận xét:** Chênh lệch giữa các thứ nhỏ hơn nhiều so với chênh lệch giữa các giờ. Thứ Hai cao nhất (~112 Wh),
Thứ Ba–Thứ Năm thấp nhất (~87–90 Wh); trung bình cuối tuần (~101 Wh) chỉ cao hơn ngày thường (~97 Wh) một chút.
→ `day_of_week` có thể có ích nhưng yếu hơn `hour`; mô hình sẽ cho biết qua feature importance.

### 2.3 Heatmap giờ × thứ

In [ ]:
pivot = dft.pivot_table(index="day_of_week", columns="hour", values=TARGET, aggfunc="mean")
pivot.index = DAY_NAMES
fig, ax = plt.subplots(figsize=(14, 4))
sns.heatmap(pivot, cmap="YlOrRd", ax=ax, cbar_kws={"label": "Wh"})
ax.set_title("Điện năng trung bình theo giờ và thứ")
ax.set_xlabel("Giờ")
ax.set_ylabel("")
fig.tight_layout()
save_fig(fig, "02_heatmap_hour_weekday")
plt.show()

**Nhận xét:** Heatmap xác nhận mô hình "ngủ – thức": dải tối ở ban đêm cho mọi thứ trong tuần, các ô nóng tập
trung vào chiều tối 17h–20h. Cuối tuần, mức tiêu thụ ban ngày (10h–15h) có xu hướng cao hơn vì gia đình ở nhà.

### 2.4 Xu hướng dài hạn theo tuần

In [ ]:
weekly = df.set_index("date")[TARGET].resample("W").mean()
fig, ax = plt.subplots(figsize=(11, 3.5))
ax.plot(weekly.index, weekly.values, marker="o")
ax.set_title("Điện năng trung bình theo tuần")
ax.set_ylabel("Appliances (Wh)")
fig.tight_layout()
save_fig(fig, "02_weekly_trend")
plt.show()

**Nhận xét:** Trung bình theo tuần dao động 75–122 Wh, **không có xu hướng tăng/giảm rõ rệt** trong 4,5 tháng.
Điều này thuận lợi cho việc chia train/test theo thời gian: phân phối tập test không lệch quá xa tập train.

## Tóm tắt EDA (phần 1–2)
| Phát hiện | Hệ quả cho mô hình |
|---|---|
| Biến mục tiêu lệch phải mạnh, ~10,8% ngoại lệ theo IQR | Không xóa ngoại lệ; thử biến đổi `log1p`; báo cáo cả MAE và RMSE |
| Chu kỳ ngày rất rõ (đỉnh 18h) | Dùng đặc trưng giờ + mã hóa sin/cos |
| Khác biệt theo thứ nhỏ | Giữ `day_of_week`, `is_weekend`; kiểm chứng bằng feature importance |
| Không có xu hướng dài hạn | Chia dữ liệu theo thời gian là hợp lý |